# GRB_Technology — Virtual Clothing Try-On

**Before running:** `Runtime → Change runtime type → GPU (T4)` then Save.

Run all 3 cells. A public Gradio URL will appear at the end.

In [ ]:
# Cell 1 — Install packages
!pip install -q gradio pillow opencv-python-headless rembg
print('Done!')

In [ ]:
# Cell 2 — Define the try-on pipeline
import cv2, numpy as np, io
from PIL import Image

def remove_bg(img_arr):
    """Remove white/light background from clothing image."""
    gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 235, 255, cv2.THRESH_BINARY_INV)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.GaussianBlur(mask, (9,9), 0)
    return mask.astype(np.float32) / 255.0

def tryon(person_pil, cloth_pil):
    if person_pil is None or cloth_pil is None:
        return None, 'Upload both images first.'
    try:
        # Resize person to standard resolution
        person = np.array(person_pil.convert('RGB').resize((384, 512), Image.LANCZOS))
        cloth  = np.array(cloth_pil.convert('RGB'))
        h, w   = person.shape[:2]

        # Estimate torso bounding box
        x1, y1 = int(w*0.10), int(h*0.17)
        x2, y2 = int(w*0.90), int(h*0.65)
        rw, rh = x2-x1, y2-y1

        # Resize cloth to fit torso region
        cloth_r = cv2.resize(cloth, (rw, rh), interpolation=cv2.INTER_LANCZOS4)

        # Build soft cloth mask
        alpha = remove_bg(cloth_r)[:,:,np.newaxis]  # [H,W,1]

        # Blend cloth onto person
        result = person.copy().astype(np.float32)
        region = result[y1:y2, x1:x2]
        region[:] = cloth_r.astype(np.float32)*alpha + region*(1-alpha)
        result = np.clip(result, 0, 255).astype(np.uint8)

        return Image.fromarray(result), 'Done!'
    except Exception as e:
        return None, f'Error: {e}'

print('Pipeline ready!')

In [ ]:
# Cell 3 — Launch Gradio UI (public URL will appear below)
import gradio as gr

with gr.Blocks(title='GRB_Technology Virtual Try-On', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# GRB_Technology — Virtual Clothing Try-On')
    gr.Markdown('Upload a full-body person photo and a clothing item on a **white background**.')
    with gr.Row():
        inp_person = gr.Image(type='pil', label='Your Photo (full-body)')
        inp_cloth  = gr.Image(type='pil', label='Clothing Item (white bg)')
        out_result = gr.Image(type='pil', label='Try-On Result')
    out_status = gr.Textbox(label='Status', interactive=False)
    with gr.Row():
        btn_run   = gr.Button('Try It On!', variant='primary')
        btn_clear = gr.Button('Clear')
    btn_run.click(fn=tryon, inputs=[inp_person, inp_cloth], outputs=[out_result, out_status])
    btn_clear.click(fn=lambda: (None,None,None,''), outputs=[inp_person,inp_cloth,out_result,out_status])
    gr.Markdown('**Tips:** Use a straight front-facing photo. Clothing on plain white background works best.')

demo.launch(share=True)